# Jump-Diffusion Models in Finance

From-scratch exploration of Poisson processes, Merton's jump-diffusion model, option pricing with jumps, and the implied volatility smile.

**Outline**
1. Why GBM isn't enough -- crashes happen!
2. The Poisson process
3. Compound Poisson process
4. Merton's jump-diffusion model
5. Return distribution -- mixture of normals
6. Merton's option pricing formula
7. Series convergence
8. The implied volatility smile
9. Monte Carlo pricing
10. Sensitivity to jump parameters
11. Calibration
12. Summary
13. References

---
## 1. Why GBM Isn't Enough -- Crashes Happen!

GBM assumes continuous price paths. Under GBM, a 20% single-day drop is essentially impossible.

But real markets disagree:
- **Black Monday (1987):** S&P 500 fell 20.5% in one day -- a 25-sigma event under GBM.
- **Flash Crash (2010):** Dow dropped 9% in minutes.
- **COVID Crash (2020):** Multiple days of 5-10% moves.

Real returns have **fat tails** -- far more extreme moves than normal predicts.

| Feature | GBM | Reality |
|---------|-----|---------|
| Price paths | Continuous | Can jump suddenly |
| Return distribution | Normal (thin tails) | Fat tails (leptokurtic) |
| Skewness | Zero | Negative (crashes > rallies) |
| Implied volatility | Flat across strikes | Smile/skew pattern |

### Merton's insight (1976)

Keep GBM for "normal" fluctuations, add occasional **jumps** for crashes/rallies. Jumps arrive randomly (like earthquakes) with random sizes.

> **Key Concept:** The jump-diffusion model separates two types of risk: (1) **diffusion risk** -- small, continuous noise, and (2) **jump risk** -- rare, sudden large moves. Together, they produce fat tails and the volatility smile.

### Real-world analogy

Stock prices are like sea level: **diffusion** = gentle waves (continuous, small), **jumps** = tsunamis (rare, sudden). You need separate mechanisms for each.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 2. The Poisson Process -- Random Arrivals

### Real-world examples
- Earthquakes (~1 major per decade)
- Customer arrivals (~5 per hour)
- Market crashes (~1 per 5-10 years)

### Definition

$$P(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}$$

**Variables:** $N(t)$ = number of events by time $t$, $\lambda$ = intensity (events per unit time).

**Key properties:** $E[N(t)] = \lambda t$, $\text{Var}[N(t)] = \lambda t$, inter-arrival times are Exp($\lambda$).

### Worked example

Crashes at rate $\lambda = 0.5$/year:
- P(0 crashes in 1 year) = $e^{-0.5} = 61\%$
- P(1 crash) = $0.5 e^{-0.5} = 30\%$
- P(2+ crashes) = $9\%$
- Expected wait: $1/\lambda = 2$ years

In [ ]:
def simulate_poisson_process(lam, T, rng):
    """Simulate a Poisson process via inter-arrival times.
    
    Args:
        lam: Intensity (expected jumps per unit time)
        T: Time horizon
        rng: NumPy random generator
    
    Returns:
        jump_times: array of jump times in [0, T]
    """
    jump_times = []
    t = 0.0
    while True:
        # Inter-arrival time ~ Exp(lambda)
        dt = rng.exponential(1.0 / lam)
        t += dt
        if t > T:
            break
        jump_times.append(t)
    return np.array(jump_times)

# Simulate and visualize
lam = 3.0  # 3 jumps per year on average
T = 5.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample paths
for i in range(5):
    jt = simulate_poisson_process(lam, T, rng)
    times = np.concatenate([[0], np.repeat(jt, 2), [T]])
    counts = np.repeat(np.arange(len(jt) + 1), 2)
    axes[0].plot(times, counts, alpha=0.7, label=f'Path {i+1}')

axes[0].set_xlabel('Time')
axes[0].set_ylabel('N(t)')
axes[0].set_title(f'Poisson Process Sample Paths ($\\lambda = {lam}$)')
axes[0].legend(fontsize=9)

# Right: distribution of N(T) vs theoretical
n_sims = 10000
N_T = rng.poisson(lam * T, size=n_sims)
k_vals = np.arange(0, max(N_T) + 1)
pmf_theory = stats.poisson.pmf(k_vals, lam * T)

axes[1].hist(N_T, bins=k_vals - 0.5, density=True, alpha=0.6, color=PRIMARY, label='Simulated')
axes[1].plot(k_vals, pmf_theory, 'o-', color=SECONDARY, label='Theoretical PMF')
axes[1].set_xlabel('k')
axes[1].set_ylabel('P(N(T) = k)')
axes[1].set_title(f'Distribution of N({T}) with $\\lambda = {lam}$')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Compound Poisson Process -- Random-Sized Jumps

$$X(t) = \sum_{i=1}^{N(t)} J_i$$

In Merton's model: $\ln(1 + J_i) \sim \mathcal{N}(\mu_J, \sigma_J^2)$ (lognormal jump factors ensure positive prices).

### Worked example

$\mu_J = -0.05$, $\sigma_J = 0.10$: average factor $e^{-0.045} = 0.956$ (about -4.4%). A 2-sigma event: stock drops 22% or rises 16%.

In [ ]:
def simulate_compound_poisson(lam, mu_J, sigma_J, T, rng):
    """Simulate a compound Poisson process with lognormal jumps.
    
    Args:
        lam: Jump intensity
        mu_J: Mean of log-jump size
        sigma_J: Std of log-jump size
        T: Time horizon
        rng: NumPy random generator
    
    Returns:
        jump_times: array of jump times
        jump_sizes: array of multiplicative jump factors (1 + J_i)
    """
    jump_times = simulate_poisson_process(lam, T, rng)
    n_jumps = len(jump_times)
    if n_jumps == 0:
        return jump_times, np.array([])
    # Log-jump sizes ~ N(mu_J, sigma_J^2)
    log_jumps = rng.normal(mu_J, sigma_J, size=n_jumps)
    jump_factors = np.exp(log_jumps)  # multiplicative: 1+J = exp(log_jump)
    return jump_times, jump_factors

# Example: compound Poisson with negative mean jumps (crash-like)
lam_ex = 2.0
mu_J_ex = -0.05   # mean log-jump: slight negative (crashes)
sigma_J_ex = 0.10  # volatility of jump size

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(5):
    jt, jf = simulate_compound_poisson(lam_ex, mu_J_ex, sigma_J_ex, 3.0, rng)
    # Build cumulative product path
    if len(jt) > 0:
        cum_jump = np.concatenate([[1.0], np.cumprod(jf)])
        times = np.concatenate([[0], jt])
    else:
        cum_jump = np.array([1.0])
        times = np.array([0])
    axes[0].step(times, cum_jump, where='post', alpha=0.7)

axes[0].set_xlabel('Time')
axes[0].set_ylabel('Cumulative Jump Factor')
axes[0].set_title('Compound Poisson Process (Multiplicative)')
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.5)

# Distribution of jump sizes
jump_samples = np.exp(rng.normal(mu_J_ex, sigma_J_ex, 10000)) - 1  # J_i values
axes[1].hist(jump_samples, bins=80, density=True, alpha=0.6, color=PRIMARY)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Jump Size $J_i$')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Jump Size Distribution ($\\mu_J = {mu_J_ex}$, $\\sigma_J = {sigma_J_ex}$)')

plt.tight_layout()
plt.show()

---
## 4. Merton's Jump-Diffusion Model

$$\frac{dS}{S} = (\mu - \lambda k) \, dt + \sigma \, dW + J \, dN$$

| Term | Meaning | Analogy |
|------|---------|---------|
| $(\mu - \lambda k) \, dt$ | Drift (adjusted) | River current |
| $\sigma \, dW$ | Diffusion | Gentle waves |
| $J \, dN$ | Jumps | Tsunamis |

**Variables:** $k = E[J] = e^{\mu_J + \sigma_J^2/2} - 1$. The $-\lambda k$ term **compensates** for the average jump, ensuring overall expected return is exactly $\mu$.

> **Key Concept:** The compensator separates "risk" of jumps from "reward." For risk-neutral pricing, expected return must be $r$ regardless of jump parameters.

In [ ]:
def simulate_merton_jump_diffusion(S0, mu, sigma, lam, mu_J, sigma_J, T, n_steps, n_paths, rng):
    """Simulate Merton jump-diffusion paths.
    
    Uses exact simulation: for each time step, add the diffusion component
    and multiply by any jumps that occur in that interval.
    
    Args:
        S0: Initial stock price
        mu: Drift
        sigma: Diffusion volatility
        lam: Jump intensity
        mu_J: Mean of log-jump size
        sigma_J: Std of log-jump size
        T: Time horizon
        n_steps: Number of time steps
        n_paths: Number of simulation paths
        rng: NumPy random generator
    
    Returns:
        t: time grid (n_steps + 1,)
        S: price paths (n_paths, n_steps + 1)
    """
    dt = T / n_steps
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1  # E[J]
    
    t = np.linspace(0, T, n_steps + 1)
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_steps):
        # Diffusion component
        Z = rng.standard_normal(n_paths)
        diffusion = (mu - lam * k - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
        
        # Jump component: number of jumps in [t_i, t_{i+1}]
        n_jumps = rng.poisson(lam * dt, size=n_paths)
        
        # Sum of log-jump sizes for each path
        jump_component = np.zeros(n_paths)
        for j in range(n_paths):
            if n_jumps[j] > 0:
                log_jumps = rng.normal(mu_J, sigma_J, size=n_jumps[j])
                jump_component[j] = np.sum(log_jumps)
        
        S[:, i+1] = S[:, i] * np.exp(diffusion + jump_component)
    
    return t, S

# Parameters
S0 = 100
mu = 0.08
sigma = 0.20
lam = 1.0        # 1 jump per year on average
mu_J = -0.10     # mean log-jump: negative (crashes)
sigma_J = 0.15   # jump size volatility
T = 2.0
n_steps = 500
n_paths = 10

t, S = simulate_merton_jump_diffusion(S0, mu, sigma, lam, mu_J, sigma_J, T, n_steps, n_paths, rng)

# Compare with pure GBM paths
rng_gbm = np.random.default_rng(SEED + 1)
dt_gbm = T / n_steps
S_gbm = np.zeros((n_paths, n_steps + 1))
S_gbm[:, 0] = S0
for i in range(n_steps):
    Z = rng_gbm.standard_normal(n_paths)
    S_gbm[:, i+1] = S_gbm[:, i] * np.exp((mu - 0.5*sigma**2)*dt_gbm + sigma*np.sqrt(dt_gbm)*Z)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for p in range(n_paths):
    axes[0].plot(t, S_gbm[p], alpha=0.5, color=PRIMARY)
axes[0].set_title('GBM Paths (No Jumps)')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Stock Price')

for p in range(n_paths):
    axes[1].plot(t, S[p], alpha=0.5, color=SECONDARY)
axes[1].set_title('Merton Jump-Diffusion Paths')
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Stock Price')

plt.tight_layout()
plt.show()

---
## 5. Return Distribution: The Mixture of Normals

Conditional on $n$ jumps:

$$\ln(S_T/S_0) | N(T)=n \sim \mathcal{N}((\mu - \lambda k - \sigma^2/2)T + n\mu_J, \sigma^2 T + n\sigma_J^2)$$

The unconditional distribution is a **Poisson mixture of normals**, producing:
- **Heavier tails** (excess kurtosis)
- **Negative skewness** (when $\mu_J < 0$)

> **Key Concept:** Mixing distributions with different variances always produces fatter tails than any single component. This is why jump-diffusion returns have excess kurtosis.

In [ ]:
# Large-scale simulation for distribution comparison
n_sims = 100000
T_dist = 1.0 / 12  # monthly returns

# Jump-diffusion terminal values
_, S_jd = simulate_merton_jump_diffusion(S0, mu, sigma, lam, mu_J, sigma_J, T_dist, 1, n_sims, rng)
log_returns_jd = np.log(S_jd[:, -1] / S0)

# GBM terminal values
Z_gbm = rng.standard_normal(n_sims)
log_returns_gbm = (mu - 0.5 * sigma**2) * T_dist + sigma * np.sqrt(T_dist) * Z_gbm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograms
bins = np.linspace(-0.30, 0.20, 100)
axes[0].hist(log_returns_gbm, bins=bins, density=True, alpha=0.5, color=PRIMARY, label='GBM')
axes[0].hist(log_returns_jd, bins=bins, density=True, alpha=0.5, color=SECONDARY, label='Jump-Diffusion')
axes[0].set_xlabel('Monthly Log Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Return Distribution Comparison')
axes[0].legend()

# QQ plot
sorted_jd = np.sort(log_returns_jd)
theoretical_quantiles = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_jd)))
# Standardize
jd_std = (sorted_jd - np.mean(sorted_jd)) / np.std(sorted_jd)
subsample = np.linspace(0, len(jd_std)-1, 1000, dtype=int)
axes[1].scatter(theoretical_quantiles[subsample], jd_std[subsample], s=2, alpha=0.5, color=SECONDARY)
axes[1].plot([-4, 4], [-4, 4], 'k--', alpha=0.5, label='Normal reference')
axes[1].set_xlabel('Theoretical Quantiles (Normal)')
axes[1].set_ylabel('Sample Quantiles (Jump-Diffusion)')
axes[1].set_title('QQ Plot: Jump-Diffusion vs Normal')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print(f"{'Statistic':<20} {'GBM':>12} {'Jump-Diffusion':>15}")
print('-' * 50)
for name, data_gbm, data_jd in [
    ('Mean', np.mean(log_returns_gbm), np.mean(log_returns_jd)),
    ('Std Dev', np.std(log_returns_gbm), np.std(log_returns_jd)),
    ('Skewness', stats.skew(log_returns_gbm), stats.skew(log_returns_jd)),
    ('Kurtosis', stats.kurtosis(log_returns_gbm), stats.kurtosis(log_returns_jd)),
]:
    print(f"{name:<20} {data_gbm:>12.4f} {data_jd:>15.4f}")

---
## 6. Merton's Option Pricing Formula

$$C = \sum_{n=0}^{\infty} \frac{e^{-\lambda' T} (\lambda' T)^n}{n!} C_{\text{BS}}(S_0, K, T, r_n, \sigma_n)$$

**Reading this:** Average over all possible numbers of jumps. Each scenario is BS-priced with:
- $\sigma_n = \sqrt{\sigma^2 + n\sigma_J^2/T}$ (jumps add variance)
- $r_n = r - \lambda k + n\ln(1+k)/T$ (rate adjustment)
- $\lambda' = \lambda(1+k)$ (risk-neutral intensity)

> **Key Concept:** Since jumps increase variance, every additional jump raises the BS price. Jump-diffusion options are generally more expensive than BS -- the market is pricing crash risk.

In [ ]:
def bs_call(S, K, T, r, sigma):
    """Standard Black-Scholes European call price."""
    if T <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def bs_put(S, K, T, r, sigma):
    """Standard Black-Scholes European put price."""
    call = bs_call(S, K, T, r, sigma)
    return call - S + K * np.exp(-r * T)

def merton_call(S, K, T, r, sigma, lam, mu_J, sigma_J, n_terms=50):
    """Merton jump-diffusion European call price.
    
    Args:
        S: Spot price
        K: Strike price
        T: Time to maturity
        r: Risk-free rate
        sigma: Diffusion volatility
        lam: Jump intensity
        mu_J: Mean of log-jump
        sigma_J: Std of log-jump
        n_terms: Number of series terms
    
    Returns:
        Call price
    """
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1  # E[J]
    lam_prime = lam * (1 + k)
    
    price = 0.0
    for n in range(n_terms):
        # Poisson weight
        poisson_weight = np.exp(-lam_prime * T) * (lam_prime * T)**n / np.math.factorial(n)
        
        # Adjusted volatility and rate
        sigma_n = np.sqrt(sigma**2 + n * sigma_J**2 / T)
        r_n = r - lam * k + n * np.log(1 + k) / T
        
        price += poisson_weight * bs_call(S, K, T, r_n, sigma_n)
    
    return price

def merton_put(S, K, T, r, sigma, lam, mu_J, sigma_J, n_terms=50):
    """Merton jump-diffusion European put via put-call parity."""
    call = merton_call(S, K, T, r, sigma, lam, mu_J, sigma_J, n_terms)
    return call - S + K * np.exp(-r * T)

# Example pricing
S0_opt = 100
K_opt = 100
T_opt = 0.5
r_opt = 0.05
sigma_opt = 0.20
lam_opt = 1.0
mu_J_opt = -0.10
sigma_J_opt = 0.15

c_bs = bs_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt)
c_merton = merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)
p_bs = bs_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt)
p_merton = merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)

print(f"{'':20} {'Black-Scholes':>14} {'Merton JD':>14} {'Difference':>12}")
print('-' * 62)
print(f"{'Call Price':20} {c_bs:>14.4f} {c_merton:>14.4f} {c_merton - c_bs:>12.4f}")
print(f"{'Put Price':20} {p_bs:>14.4f} {p_merton:>14.4f} {p_merton - p_bs:>12.4f}")

---
## 7. Series Convergence

The Poisson weights decay factorially, so 5-8 terms suffice for machine precision.

In [ ]:
# Convergence of Merton series
max_terms = 30
partial_sums = []

k_comp = np.exp(mu_J_opt + 0.5 * sigma_J_opt**2) - 1
lam_p = lam_opt * (1 + k_comp)

running_sum = 0.0
for n in range(max_terms):
    pw = np.exp(-lam_p * T_opt) * (lam_p * T_opt)**n / np.math.factorial(n)
    sigma_n = np.sqrt(sigma_opt**2 + n * sigma_J_opt**2 / T_opt)
    r_n = r_opt - lam_opt * k_comp + n * np.log(1 + k_comp) / T_opt
    running_sum += pw * bs_call(S0_opt, K_opt, T_opt, r_n, sigma_n)
    partial_sums.append(running_sum)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(max_terms), partial_sums, 'o-', color=PRIMARY, markersize=4)
axes[0].axhline(y=partial_sums[-1], color=SECONDARY, linestyle='--', label=f'Converged: {partial_sums[-1]:.4f}')
axes[0].set_xlabel('Number of Terms')
axes[0].set_ylabel('Call Price')
axes[0].set_title('Merton Series Convergence')
axes[0].legend()

# Error vs number of terms
errors = np.abs(np.array(partial_sums) - partial_sums[-1])
errors[errors == 0] = 1e-16  # avoid log(0)
axes[1].semilogy(range(max_terms), errors, 'o-', color=SECONDARY, markersize=4)
axes[1].set_xlabel('Number of Terms')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Convergence Error (Log Scale)')

plt.tight_layout()
plt.show()

---
## 8. The Implied Volatility Smile

### Why jumps produce the smile

In the BS world, implied vol should be constant across strikes. Jumps create a **smile/skew**:

1. **OTM puts (low strikes):** Crashes more likely with jumps, so priced higher.
2. **ATM options:** Jump component less important, implied vol near $\sigma$.
3. **OTM calls (high strikes):** Positive jumps also more likely.

> **Key Concept:** The volatility smile is not a "market anomaly" -- it correctly prices jump risk. It says: "the real world has fatter tails than Black-Scholes assumes."

| Parameter change | Effect on smile |
|-----------------|----------------|
| More negative $\mu_J$ | Steeper skew (left side rises) |
| Higher $\sigma_J$ | More symmetric smile |
| Higher $\lambda$ | Smile more pronounced |

In [ ]:
def implied_vol_bisection(market_price, S, K, T, r, option_type='call', tol=1e-8, max_iter=200):
    """Find implied volatility using bisection method."""
    low, high = 0.001, 3.0
    price_fn = bs_call if option_type == 'call' else bs_put
    
    for _ in range(max_iter):
        mid = (low + high) / 2
        price = price_fn(S, K, T, r, mid)
        if abs(price - market_price) < tol:
            return mid
        if price > market_price:
            high = mid
        else:
            low = mid
    return mid

# Compute Merton prices across strikes, then back out implied vol
strikes = np.linspace(70, 130, 50)
iv_smile = []

for K in strikes:
    # Merton call price
    c_merton_k = merton_call(S0_opt, K, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)
    # Back out implied vol
    iv = implied_vol_bisection(c_merton_k, S0_opt, K, T_opt, r_opt)
    iv_smile.append(iv)

# Different jump parameters for comparison
params_list = [
    (1.0, -0.10, 0.15, 'Negative jumps ($\\mu_J=-0.10$)'),
    (1.0,  0.00, 0.15, 'Symmetric jumps ($\\mu_J=0$)'),
    (1.0,  0.05, 0.15, 'Positive jumps ($\\mu_J=0.05$)'),
    (3.0, -0.05, 0.10, 'Frequent small crashes'),
]

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PRIMARY, SECONDARY, TERTIARY, ACCENT]

for (l, mj, sj, label), color in zip(params_list, colors):
    ivs = []
    for K in strikes:
        c = merton_call(S0_opt, K, T_opt, r_opt, sigma_opt, l, mj, sj)
        ivs.append(implied_vol_bisection(c, S0_opt, K, T_opt, r_opt))
    ax.plot(strikes / S0_opt, np.array(ivs) * 100, label=label, color=color, linewidth=2)

ax.axhline(y=sigma_opt * 100, color='black', linestyle='--', alpha=0.5, label='BSM flat vol')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Moneyness (K/S)')
ax.set_ylabel('Implied Volatility (%)')
ax.set_title('Implied Volatility Smile from Merton Jump-Diffusion')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 9. Monte Carlo Pricing Under Jump-Diffusion

For path-dependent options where Merton's formula doesn't apply:

$$C \approx e^{-rT} \frac{1}{M}\sum_{m=1}^{M} \text{payoff}(S_T^{(m)})$$

> **Key Concept:** Monte Carlo converges at $O(1/\sqrt{M})$ -- to halve the error, you need 4x more paths. Slow but universal.

In [ ]:
def mc_european_jump_diffusion(S0, K, T, r, sigma, lam, mu_J, sigma_J, n_paths, rng, option_type='call'):
    """Monte Carlo pricing of European options under jump-diffusion.
    
    Uses exact terminal distribution (single step).
    """
    k = np.exp(mu_J + 0.5 * sigma_J**2) - 1
    
    # Number of jumps for each path
    n_jumps = rng.poisson(lam * T, size=n_paths)
    
    # Diffusion component
    Z = rng.standard_normal(n_paths)
    diffusion = (r - lam * k - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z
    
    # Jump component
    jump_component = np.zeros(n_paths)
    for i in range(n_paths):
        if n_jumps[i] > 0:
            jump_component[i] = np.sum(rng.normal(mu_J, sigma_J, size=n_jumps[i]))
    
    S_T = S0 * np.exp(diffusion + jump_component)
    
    if option_type == 'call':
        payoffs = np.maximum(S_T - K, 0)
    else:
        payoffs = np.maximum(K - S_T, 0)
    
    price = np.exp(-r * T) * np.mean(payoffs)
    se = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_paths)
    return price, se

# Compare MC with analytical Merton formula
path_counts = [1000, 5000, 10000, 50000, 100000, 500000]
mc_prices = []
mc_errors = []
analytical = merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt)

for n in path_counts:
    p, se = mc_european_jump_diffusion(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sigma_J_opt, n, rng)
    mc_prices.append(p)
    mc_errors.append(se)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].errorbar(path_counts, mc_prices, yerr=[1.96*se for se in mc_errors], 
                 fmt='o-', color=PRIMARY, capsize=4, label='MC estimate ± 95% CI')
axes[0].axhline(y=analytical, color=SECONDARY, linestyle='--', label=f'Analytical: {analytical:.4f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of Paths')
axes[0].set_ylabel('Call Price')
axes[0].set_title('MC Convergence to Analytical Merton Price')
axes[0].legend()

axes[1].loglog(path_counts, mc_errors, 'o-', color=SECONDARY, label='Standard Error')
# O(1/sqrt(N)) reference
ref = mc_errors[0] * np.sqrt(path_counts[0]) / np.sqrt(path_counts)
axes[1].loglog(path_counts, ref, '--', color='gray', label='$O(1/\\sqrt{N})$')
axes[1].set_xlabel('Number of Paths')
axes[1].set_ylabel('Standard Error')
axes[1].set_title('MC Standard Error Convergence')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Sensitivity to Jump Parameters

| Parameter | Increase causes... |
|-----------|-------------------|
| $\lambda$ | More frequent jumps -- both calls and puts increase |
| $\mu_J$ | More negative: puts rise (crash risk), calls may fall |
| $\sigma_J$ | Fatter tails -- both increase |

> **Key Concept:** Jump risk is fundamentally **non-hedgeable** with the underlying alone. Unlike diffusion risk (delta-hedged), jumps are sudden. The market charges a "jump risk premium."

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sensitivity to lambda (jump intensity)
lam_range = np.linspace(0, 5, 30)
calls_lam = [merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, l, mu_J_opt, sigma_J_opt) for l in lam_range]
puts_lam = [merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, l, mu_J_opt, sigma_J_opt) for l in lam_range]

axes[0].plot(lam_range, calls_lam, color=PRIMARY, linewidth=2, label='Call')
axes[0].plot(lam_range, puts_lam, color=SECONDARY, linewidth=2, label='Put')
axes[0].axhline(y=c_bs, color=PRIMARY, linestyle='--', alpha=0.4, label='BS Call')
axes[0].axhline(y=p_bs, color=SECONDARY, linestyle='--', alpha=0.4, label='BS Put')
axes[0].set_xlabel('Jump Intensity $\\lambda$')
axes[0].set_ylabel('Option Price')
axes[0].set_title('Sensitivity to $\\lambda$')
axes[0].legend(fontsize=9)

# Sensitivity to mu_J (mean jump size)
mj_range = np.linspace(-0.30, 0.10, 30)
calls_mj = [merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mj, sigma_J_opt) for mj in mj_range]
puts_mj = [merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mj, sigma_J_opt) for mj in mj_range]

axes[1].plot(mj_range, calls_mj, color=PRIMARY, linewidth=2, label='Call')
axes[1].plot(mj_range, puts_mj, color=SECONDARY, linewidth=2, label='Put')
axes[1].set_xlabel('Mean Log-Jump $\\mu_J$')
axes[1].set_ylabel('Option Price')
axes[1].set_title('Sensitivity to $\\mu_J$')
axes[1].legend(fontsize=9)

# Sensitivity to sigma_J (jump volatility)
sj_range = np.linspace(0.01, 0.40, 30)
calls_sj = [merton_call(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sj) for sj in sj_range]
puts_sj = [merton_put(S0_opt, K_opt, T_opt, r_opt, sigma_opt, lam_opt, mu_J_opt, sj) for sj in sj_range]

axes[2].plot(sj_range, calls_sj, color=PRIMARY, linewidth=2, label='Call')
axes[2].plot(sj_range, puts_sj, color=SECONDARY, linewidth=2, label='Put')
axes[2].set_xlabel('Jump Volatility $\\sigma_J$')
axes[2].set_ylabel('Option Price')
axes[2].set_title('Sensitivity to $\\sigma_J$')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 11. Calibration to Market Data

Minimize pricing error across strikes: $\min_{\sigma, \lambda, \mu_J, \sigma_J} \sum_i (C_i^{\text{model}} - C_i^{\text{market}})^2$

In [ ]:
# Generate synthetic market prices from known parameters
true_params = {'sigma': 0.20, 'lam': 1.5, 'mu_J': -0.08, 'sigma_J': 0.12}
calib_strikes = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120])
calib_T = 0.5
calib_r = 0.05

market_prices = np.array([
    merton_call(S0_opt, K, calib_T, calib_r, true_params['sigma'],
                true_params['lam'], true_params['mu_J'], true_params['sigma_J'])
    for K in calib_strikes
])

# Add small noise to simulate real market data
market_prices += rng.normal(0, 0.05, len(market_prices))

def calibration_objective(params):
    """Sum of squared pricing errors."""
    sigma, lam, mu_J, sigma_J = params
    if sigma <= 0 or lam < 0 or sigma_J <= 0:
        return 1e10
    model_prices = np.array([
        merton_call(S0_opt, K, calib_T, calib_r, sigma, lam, mu_J, sigma_J)
        for K in calib_strikes
    ])
    return np.sum((model_prices - market_prices)**2)

# Calibrate using Nelder-Mead
x0 = [0.25, 1.0, -0.05, 0.10]  # initial guess
result = optimize.minimize(calibration_objective, x0, method='Nelder-Mead',
                           options={'xatol': 1e-8, 'fatol': 1e-10, 'maxiter': 5000})

sigma_cal, lam_cal, mu_J_cal, sigma_J_cal = result.x

print(f"{'Parameter':<15} {'True':>10} {'Calibrated':>12}")
print('-' * 40)
for name, true_val, cal_val in [
    ('sigma', true_params['sigma'], sigma_cal),
    ('lambda', true_params['lam'], lam_cal),
    ('mu_J', true_params['mu_J'], mu_J_cal),
    ('sigma_J', true_params['sigma_J'], sigma_J_cal),
]:
    print(f"{name:<15} {true_val:>10.4f} {cal_val:>12.4f}")

# Plot fit
fitted_prices = [merton_call(S0_opt, K, calib_T, calib_r, sigma_cal, lam_cal, mu_J_cal, sigma_J_cal)
                 for K in calib_strikes]

plt.figure(figsize=(10, 6))
plt.plot(calib_strikes, market_prices, 'o', color=SECONDARY, markersize=8, label='Market prices')
plt.plot(calib_strikes, fitted_prices, 's-', color=PRIMARY, markersize=6, label='Calibrated model')
plt.xlabel('Strike Price')
plt.ylabel('Call Price')
plt.title('Merton Jump-Diffusion Calibration')
plt.legend()
plt.tight_layout()
plt.show()

---
## 12. Summary & Extensions

| Concept | Key Result |
|---------|------------|
| Poisson process | Models rare, discrete events with intensity $\lambda$ |
| Merton JD | $dS/S = (\mu - \lambda k)dt + \sigma dW + J dN$ |
| Return distribution | Poisson mixture of normals -- heavy tails |
| Option pricing | Series of BS prices weighted by Poisson PMF |
| Implied volatility | Naturally produces smile/skew |
| Jump risk | Non-hedgeable with underlying alone |

## 13. References

1. Merton, R. C. "Option Pricing When Underlying Stock Returns Are Discontinuous," *JFE*, 1976.
2. Black, F. & Scholes, M. "The Pricing of Options and Corporate Liabilities," *JPE*, 1973.
3. Bates, D. "Jumps and Stochastic Volatility," *RFS*, 1996.
4. Cont, R. & Tankov, P. *Financial Modelling with Jump Processes*, CRC Press, 2003.
5. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed., Pearson, 2022.
6. Glasserman, P. *Monte Carlo Methods in Financial Engineering*, Springer, 2003.